In [1]:
import os

# ============================================================
# Part 1. Define model directory
# ============================================================

save_dir = (
    r"C:\Users\65163\Desktop\python"
    r"\NHANES_Diabetes\BorutaShap_Saved_Models"
)

# ============================================================
# Part 2. List all saved files
# ============================================================

print("=" * 60)
print("SAVED MODEL FILES")
print("=" * 60)

for file_name in sorted(os.listdir(save_dir)):
    file_path = os.path.join(save_dir, file_name)

    if os.path.isfile(file_path):
        print(
            f"{file_name:<45}"
            f"{os.path.getsize(file_path) / 1024:.2f} KB"
        )

# ============================================================
# Part 3. Check directory
# ============================================================

print("\nDirectory exists:", os.path.exists(save_dir))

SAVED MODEL FILES
BorutaShap_feature_info.joblib               0.14 KB
CNN_BorutaShap.keras                         87.36 KB
CNN_classes.joblib                           0.24 KB
CNN_preprocessor.joblib                      2.56 KB
GRU_BorutaShap.keras                         55.01 KB
GRU_classes.joblib                           0.24 KB
GRU_preprocessor.joblib                      2.56 KB
KAN_BorutaShap_cloudpickle.pkl               888.57 KB
KAN_classes.joblib                           0.24 KB
KAN_preprocessor.joblib                      2.56 KB
MLP_BorutaShap.keras                         274.69 KB
MLP_classes.joblib                           0.24 KB
MLP_preprocessor.joblib                      2.56 KB

Directory exists: True


In [2]:

import os
import joblib
import cloudpickle
import tensorflow as tf

# ============================================================
# Part 1. Define directory
# ============================================================

save_dir = (
    r"C:\Users\65163\Desktop\python"
    r"\NHANES_Diabetes\BorutaShap_Saved_Models"
)

# ============================================================
# Part 2. Load MLP, CNN and GRU models
# ============================================================

models_deep_boruta = {}

for model_name in ["MLP", "CNN", "GRU"]:

    model_path = os.path.join(
        save_dir,
        f"{model_name}_BorutaShap.keras"
    )

    models_deep_boruta[model_name] = (
        tf.keras.models.load_model(
            model_path,
            compile=False
        )
    )

    print(f"{model_name} loaded successfully.")


# ============================================================
# Part 3. Load KAN model
# ============================================================

kan_path = os.path.join(
    save_dir,
    "KAN_BorutaShap_cloudpickle.pkl"
)

with open(kan_path, "rb") as f:
    models_deep_boruta["KAN"] = cloudpickle.load(f)

models_deep_boruta["KAN"].eval()

print("KAN loaded successfully.")


# ============================================================
# Part 4. Load preprocessors
# ============================================================

preprocessors_deep_boruta = {}

for model_name in ["MLP", "CNN", "GRU", "KAN"]:

    preprocessor_path = os.path.join(
        save_dir,
        f"{model_name}_preprocessor.joblib"
    )

    preprocessors_deep_boruta[model_name] = (
        joblib.load(preprocessor_path)
    )

    print(f"{model_name} preprocessor loaded successfully.")


# ============================================================
# Part 5. Load class labels
# ============================================================

classes_deep_boruta = {}

for model_name in ["MLP", "CNN", "GRU", "KAN"]:

    classes_path = os.path.join(
        save_dir,
        f"{model_name}_classes.joblib"
    )

    classes_deep_boruta[model_name] = (
        joblib.load(classes_path)
    )

    print(
        f"{model_name} classes:",
        classes_deep_boruta[model_name]
    )


# ============================================================
# Part 6. Load BorutaShap feature information
# ============================================================

feature_info_deep_boruta = joblib.load(
    os.path.join(
        save_dir,
        "BorutaShap_feature_info.joblib"
    )
)

print("\nFeature information:")
print(feature_info_deep_boruta)


# ============================================================
# Part 7. Check model input dimensions
# ============================================================

print("\n" + "=" * 60)
print("MODEL LOADING VALIDATION")
print("=" * 60)

for model_name in ["MLP", "CNN", "GRU"]:

    print(
        f"{model_name} input shape:",
        models_deep_boruta[model_name].input_shape
    )

print(
    "KAN model width:",
    models_deep_boruta["KAN"].width
)

print(
    "KAN model grid:",
    models_deep_boruta["KAN"].grid
)

print("\nAll models and preprocessors loaded successfully.")


MLP loaded successfully.
CNN loaded successfully.
GRU loaded successfully.
KAN loaded successfully.
MLP preprocessor loaded successfully.
CNN preprocessor loaded successfully.
GRU preprocessor loaded successfully.
KAN preprocessor loaded successfully.
MLP classes: [0 1]
CNN classes: [0 1]
GRU classes: [0 1]
KAN classes: [0 1]

Feature information:
{'original_features': ['BMXWAIST', 'RIDAGEYR', 'LBXIN', 'SBP_MEAN', 'LBDHDD', 'LBXRBCSI', 'BPQ020', 'LBXSGTSI'], 'transformed_dimension': 9}

MODEL LOADING VALIDATION
MLP input shape: (None, 9)
CNN input shape: (None, 9, 1)
GRU input shape: (None, 9, 1)
KAN model width: [[9, 0], [16, 0], [1, 0]]
KAN model grid: 5

All models and preprocessors loaded successfully.


In [3]:
import os
import numpy as np
import pandas as pd

# ============================================================
# 1. File paths
# ============================================================

base_dir = (
    r"C:\Users\65163\Desktop\python"
    r"\NHANES_Diabetes\2021-2023"
)

input_file = os.path.join(
    base_dir,
    "NHANES_2021_2023_BorutaShap_External_Validation.xlsx"
)

output_file = os.path.join(
    base_dir,
    "NHANES_2021_2023_BorutaShap_External_Validation_Results.xlsx"
)

# ============================================================
# 2. Load external validation data
# ============================================================

df_external = pd.read_excel(input_file)

# Use the exact original training feature order
feature_columns = feature_info_deep_boruta["original_features"]

target_column = "Diabetes"

X_external = df_external[feature_columns].copy()
y_external = df_external[target_column].astype(int).to_numpy()

# ============================================================
# 3. Basic checks
# ============================================================

assert X_external.shape[1] == 8
assert len(X_external) == len(y_external)
assert not X_external.isnull().any().any()
assert set(np.unique(y_external)).issubset({0, 1})

print("=" * 65)
print("NHANES 2021-2023 EXTERNAL VALIDATION")
print("=" * 65)

print("Sample size:", len(X_external))
print("Number of features:", X_external.shape[1])
print("Feature order:", feature_columns)

print("\nTarget distribution:")
print(pd.Series(y_external).value_counts().sort_index())

print("\nTarget percentage:")
print(
    pd.Series(y_external)
    .value_counts(normalize=True)
    .sort_index() * 100
)

print("\nData check completed.")

NHANES 2021-2023 EXTERNAL VALIDATION
Sample size: 3034
Number of features: 8
Feature order: ['BMXWAIST', 'RIDAGEYR', 'LBXIN', 'SBP_MEAN', 'LBDHDD', 'LBXRBCSI', 'BPQ020', 'LBXSGTSI']

Target distribution:
0    1163
1    1871
Name: count, dtype: int64

Target percentage:
0    38.332235
1    61.667765
Name: proportion, dtype: float64

Data check completed.


In [4]:

import torch
import tensorflow as tf
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 1. Define KAN prediction function
# ============================================================

def kan_predict_proba(model_deep, X_deep):

    model_deep.eval()

    with torch.no_grad():

        X_tensor_deep = torch.tensor(
            X_deep,
            dtype=torch.float32
        )

        logits_deep = model_deep(X_tensor_deep)

        probabilities_deep = torch.sigmoid(logits_deep)

    return probabilities_deep.cpu().numpy().reshape(-1)


# ============================================================
# 2. Initialize results
# ============================================================

external_results = []

external_predictions = {}


# ============================================================
# 3. External validation for each model
# ============================================================

for model_name in ["MLP", "CNN", "GRU", "KAN"]:

    print("\n" + "=" * 65)
    print(f"Evaluating {model_name}")
    print("=" * 65)


    # --------------------------------------------------------
    # A. Load corresponding preprocessor
    # --------------------------------------------------------

    preprocessor = preprocessors_deep_boruta[model_name]

    # Transform only, do not fit again

    X_transformed = preprocessor.transform(X_external)

    X_transformed = np.asarray(
        X_transformed,
        dtype=np.float32
    )

    print("Transformed feature shape:", X_transformed.shape)


    # --------------------------------------------------------
    # B. Prepare model-specific input
    # --------------------------------------------------------

    if model_name == "MLP":

        X_model = X_transformed

        model = models_deep_boruta["MLP"]

        y_prob = model.predict(
            X_model,
            verbose=0
        ).reshape(-1)


    elif model_name == "CNN":

        X_model = X_transformed.reshape(
            X_transformed.shape[0],
            X_transformed.shape[1],
            1
        )

        model = models_deep_boruta["CNN"]

        y_prob = model.predict(
            X_model,
            verbose=0
        ).reshape(-1)


    elif model_name == "GRU":

        X_model = X_transformed.reshape(
            X_transformed.shape[0],
            X_transformed.shape[1],
            1
        )

        model = models_deep_boruta["GRU"]

        y_prob = model.predict(
            X_model,
            verbose=0
        ).reshape(-1)


    elif model_name == "KAN":

        X_model = X_transformed

        model = models_deep_boruta["KAN"]

        y_prob = kan_predict_proba(
            model,
            X_model
        )


    # --------------------------------------------------------
    # C. Check predicted probabilities
    # --------------------------------------------------------

    if len(y_prob) != len(y_external):

        raise ValueError(
            f"{model_name}: prediction length does not match "
            "the external validation sample size."
        )


    if not np.all(np.isfinite(y_prob)):

        raise ValueError(
            f"{model_name}: predicted probabilities contain "
            "NaN or infinite values."
        )


    if np.any((y_prob < 0) | (y_prob > 1)):

        raise ValueError(
            f"{model_name}: predictions are outside [0, 1]."
        )


    # --------------------------------------------------------
    # D. Bootstrap evaluation (1000 resamples)
    # --------------------------------------------------------

    y_true_external = np.asarray(y_external).ravel()

    y_pred_external = (y_prob >= 0.5).astype(int)

    n_samples_external = len(y_true_external)

    rng_external = np.random.default_rng(50)

    metrics_external = {
        "Accuracy": [],
        "Precision": [],
        "Recall": [],
        "F1": [],
        "AUC": []
    }


    # Original estimates

    original_metrics_external = {

        "Accuracy": accuracy_score(
            y_true_external,
            y_pred_external
        ),

        "Precision": precision_score(
            y_true_external,
            y_pred_external,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true_external,
            y_pred_external,
            zero_division=0
        ),

        "F1": f1_score(
            y_true_external,
            y_pred_external,
            zero_division=0
        ),

        "AUC": roc_auc_score(
            y_true_external,
            y_prob
        )
    }


    # 1000 Bootstrap resamples

    for i in range(1000):

        indices_external = rng_external.choice(
            n_samples_external,
            size=n_samples_external,
            replace=True
        )

        y_true_boot_external = y_true_external[
            indices_external
        ]

        y_prob_boot_external = y_prob[
            indices_external
        ]

        y_pred_boot_external = (
            y_prob_boot_external >= 0.5
        ).astype(int)


        # Skip AUC calculation if only one class is present

        if len(np.unique(y_true_boot_external)) < 2:

            continue


        metrics_external["Accuracy"].append(
            accuracy_score(
                y_true_boot_external,
                y_pred_boot_external
            )
        )

        metrics_external["Precision"].append(
            precision_score(
                y_true_boot_external,
                y_pred_boot_external,
                zero_division=0
            )
        )

        metrics_external["Recall"].append(
            recall_score(
                y_true_boot_external,
                y_pred_boot_external,
                zero_division=0
            )
        )

        metrics_external["F1"].append(
            f1_score(
                y_true_boot_external,
                y_pred_boot_external,
                zero_division=0
            )
        )

        metrics_external["AUC"].append(
            roc_auc_score(
                y_true_boot_external,
                y_prob_boot_external
            )
        )


    # Calculate 95% confidence intervals

    bootstrap_results_external = []

    for metric_external in metrics_external:

        bootstrap_values_external = np.asarray(
            metrics_external[metric_external]
        )

        lower_95CI_external = np.percentile(
            bootstrap_values_external,
            2.5
        )

        upper_95CI_external = np.percentile(
            bootstrap_values_external,
            97.5
        )

        bootstrap_results_external.append({

            "Metric": metric_external,

            "Estimate": original_metrics_external[
                metric_external
            ],

            "Lower_95CI": lower_95CI_external,

            "Upper_95CI": upper_95CI_external,

            "Valid_Bootstrap": len(
                bootstrap_values_external
            )

        })


    bootstrap_results_external = pd.DataFrame(
        bootstrap_results_external
    )


    # --------------------------------------------------------
    # E. Store Bootstrap results
    # --------------------------------------------------------

    result_external = {

        "Model": model_name

    }


    for _, row_external in bootstrap_results_external.iterrows():

        result_external[row_external["Metric"]] = (

            f"{row_external['Estimate']:.3f} "
            f"({row_external['Lower_95CI']:.3f}–"
            f"{row_external['Upper_95CI']:.3f})"

        )


    result_external["Valid_Bootstrap"] = (

        bootstrap_results_external[
            "Valid_Bootstrap"
        ].iloc[0]

    )


    external_results.append(
        result_external
    )


    # --------------------------------------------------------
    # F. Store predictions
    # --------------------------------------------------------

    y_pred = (y_prob >= 0.5).astype(int)

    external_predictions[model_name] = {

        "Predicted_Label": y_pred,

        "Predicted_Probability": y_prob

    }


    print("\nBootstrap Evaluation Results:")

    print(bootstrap_results_external.round(4).to_string(index=False))


# ============================================================
# 4. Final external validation results
# ============================================================

external_results_df = pd.DataFrame(
    external_results
)


external_results_df = external_results_df[
    [
        "Model",
        "Accuracy",
        "Recall",
        "Precision",
        "F1",
        "AUC",
        "Valid_Bootstrap"
    ]
]


external_results_df = external_results_df.rename(
    columns={
        "Valid_Bootstrap": "Bootstrap"
    }
)


print(
    "\nEXTERNAL VALIDATION RESULTS (1000 Bootstrap, 95% CI)"
)

display(external_results_df)


Evaluating MLP
Transformed feature shape: (3034, 9)

Bootstrap Evaluation Results:
   Metric  Estimate  Lower_95CI  Upper_95CI  Valid_Bootstrap
 Accuracy    0.7462      0.7310      0.7611             1000
Precision    0.7724      0.7550      0.7906             1000
   Recall    0.8343      0.8182      0.8501             1000
       F1    0.8022      0.7884      0.8154             1000
      AUC    0.8083      0.7917      0.8233             1000

Evaluating CNN
Transformed feature shape: (3034, 9)

Bootstrap Evaluation Results:
   Metric  Estimate  Lower_95CI  Upper_95CI  Valid_Bootstrap
 Accuracy    0.7439      0.7287      0.7588             1000
Precision    0.7429      0.7258      0.7600             1000
   Recall    0.8942      0.8807      0.9071             1000
       F1    0.8115      0.7991      0.8243             1000
      AUC    0.8081      0.7921      0.8220             1000

Evaluating GRU
Transformed feature shape: (3034, 9)

Bootstrap Evaluation Results:
   Metric  Estim

,Model,Accuracy,Recall,Precision,F1,AUC,Bootstrap
0,MLP,0.746 (0.731–0.761),0.834 (0.818–0.850),0.772 (0.755–0.791),0.802 (0.788–0.815),0.808 (0.792–0.823),1000
1,CNN,0.744 (0.729–0.759),0.894 (0.881–0.907),0.743 (0.726–0.760),0.812 (0.799–0.824),0.808 (0.792–0.822),1000
2,GRU,0.733 (0.718–0.749),0.885 (0.871–0.899),0.736 (0.719–0.755),0.804 (0.791–0.817),0.790 (0.774–0.805),1000
3,KAN,0.748 (0.733–0.763),0.904 (0.890–0.916),0.743 (0.726–0.760),0.815 (0.803–0.828),0.814 (0.798–0.828),1000
